# Agentic AI Fundamentals

## Notebook 06: Agent State, Memory, and Context Management

An agent needs information from earlier steps in order to behave coherently, but “memory” is often used as a vague label for several different mechanisms. A tool observation needed during the current run is not the same as a user preference preserved across conversations, and neither is the same as information stored in a model's parameters.

This notebook separates those concepts and builds a small conversation-state layer. We will keep an inspectable local history, store selected preferences as structured memory, continue a real Gemini interaction across turns, and introduce simple context-management policies.

## Learning objectives

By the end of this notebook, you should be able to:

- distinguish run state, conversation history, and persistent memory;
- maintain application-owned conversation state;
- continue a Gemini conversation with `previous_interaction_id`;
- store selected facts separately from raw chat messages;
- build a bounded recent-message window;
- estimate context growth and summarize older history; and
- reset conversation state without accidentally retaining sensitive data.

## 1. Four meanings of “memory”

| Concept | Lifetime | Example |
|---|---|---|
| Run state | One agent execution | Tool calls and observations used to answer one request |
| Conversation history | One thread or session | Earlier user and assistant messages |
| Structured memory | Application-defined | Preferred answer style or selected project settings |
| Persistent memory | Across sessions | Data stored in a database or durable service |

Model parameters are sometimes described as long-term knowledge, but an application cannot reliably write a new user preference into those parameters during an ordinary conversation. If information must be available later, the application needs an explicit storage and retrieval mechanism.

## 2. Install dependencies and configure Gemini

This notebook remains self-contained for Colab. Add `GEMINI_API_KEY` through Colab Secrets before running the live sections.

In [ ]:
%pip install -q -U "google-genai>=1.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 7.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.1 which is incompatible.


In [ ]:
import os


def load_gemini_api_key() -> str:
    try:
        from google.colab import userdata

        api_key = userdata.get("GEMINI_API_KEY")
    except (ImportError, KeyError):
        api_key = os.getenv("GEMINI_API_KEY")

    if not api_key:
        raise RuntimeError(
            "Add GEMINI_API_KEY to Colab Secrets or set it as an environment variable."
        )

    return api_key


GEMINI_API_KEY = load_gemini_api_key()
print("API key loaded.")

API key loaded.


In [ ]:
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Literal
from uuid import uuid4


MODEL_NAME = "gemini-3.8-flash"
print("Configured model:", MODEL_NAME)

Configured model: gemini-3.8-flash


In [ ]:
from google import genai


client = genai.Client(api_key=GEMINI_API_KEY)
print("Gemini client initialized.")

Gemini client initialized.


## 3. Represent messages and conversation state

The application should not depend on provider response objects as its only source of state. A small local representation makes the conversation inspectable and gives us a stable place to attach timestamps, preferences, summaries, and trace metadata.

In [ ]:
Role = Literal["user", "assistant", "tool", "system"]


@dataclass(frozen=True)
class Message:
    role: Role
    content: str
    created_at: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )


@dataclass
class ConversationState:
    conversation_id: str = field(default_factory=lambda: uuid4().hex)
    previous_interaction_id: str | None = None
    messages: list[Message] = field(default_factory=list)
    preferences: dict[str, str] = field(default_factory=dict)
    summary: str | None = None
    metadata: dict[str, Any] = field(default_factory=dict)

    def add_message(self, role: Role, content: str) -> None:
        self.messages.append(Message(role=role, content=content))

    def remember(self, key: str, value: str) -> None:
        self.preferences[key] = value

    def forget(self, key: str) -> None:
        self.preferences.pop(key, None)

    def recent_messages(self, limit: int = 6) -> list[Message]:
        if limit < 1:
            raise ValueError("limit must be at least 1")
        return self.messages[-limit:]

    def reset_thread(self, keep_preferences: bool = True) -> None:
        self.previous_interaction_id = None
        self.messages.clear()
        self.summary = None

        if not keep_preferences:
            self.preferences.clear()

`previous_interaction_id` belongs to the provider conversation, while `conversation_id` belongs to our application. Keeping both avoids confusing an external API identifier with the identity and lifecycle rules of our own system.

## 4. Store selected information deliberately

A robust memory system should not automatically treat every user sentence as a permanent fact. Here, application code explicitly decides which information becomes structured memory.

In [ ]:
state = ConversationState()

state.add_message(
    "user",
    "I prefer concise explanations with one practical example.",
)
state.remember(
    "answer_style",
    "Concise explanations with one practical example",
)

print("Conversation ID:", state.conversation_id)
print("Preferences    :", state.preferences)
print("Messages       :", state.messages)

Conversation ID: 3624c116f5564f309c1f9b239fbea80d
Preferences    : {'answer_style': 'Concise explanations with one practical example'}
Messages       : [Message(role='user', content='I prefer concise explanations with one practical example.', created_at='2026-09-25T07:06:34.589636+00:00')]


This separation allows us to remove raw history while preserving an approved preference, or delete the preference without rewriting earlier messages. It also creates a natural location for consent and retention policies in a production system.

## 5. Build controlled context from memory

Structured memory must be converted into context before it can influence a model response. We will format preferences as clearly labeled application context rather than pretending they are part of the user's latest message.

In [ ]:
def build_memory_context(state: ConversationState) -> str:
    sections = []

    if state.preferences:
        preference_lines = [
            f"- {key}: {value}"
            for key, value in sorted(state.preferences.items())
        ]
        sections.append(
            "Application-managed user preferences:\n"
            + "\n".join(preference_lines)
        )

    if state.summary:
        sections.append("Summary of older conversation:\n" + state.summary)

    return "\n\n".join(sections)


print(build_memory_context(state))

Application-managed user preferences:
- answer_style: Concise explanations with one practical example


Memory should be treated as data, not as authority. A stored preference must not be allowed to override security rules, tool permissions, or system policies. In a production prompt, memory content should be clearly delimited and described as potentially untrusted user-related context.

## 6. Continue a real Gemini interaction across turns

The first API turn creates an interaction. Later turns pass its ID through `previous_interaction_id`, allowing Gemini to continue from the prior context. At the same time, we record user-visible messages locally.

In [ ]:
def chat_turn(state: ConversationState, user_message: str) -> str:
    memory_context = build_memory_context(state)
    model_input = user_message

    if memory_context:
        model_input = (
            f"{memory_context}\n\n"
            "Current user message:\n"
            f"{user_message}"
        )

    request_arguments: dict[str, Any] = {
        "model": MODEL_NAME,
        "input": model_input,
    }

    if state.previous_interaction_id is not None:
        request_arguments["previous_interaction_id"] = (
            state.previous_interaction_id
        )

    interaction = client.interactions.create(**request_arguments)
    answer = interaction.output_text

    if not answer:
        raise RuntimeError("Gemini returned no textual answer")

    state.add_message("user", user_message)
    state.add_message("assistant", answer)
    state.previous_interaction_id = interaction.id
    state.metadata["last_model"] = MODEL_NAME
    state.metadata["last_interaction_id"] = interaction.id

    return answer

In [ ]:
first_answer = chat_turn(
    state,
    "Acknowledge my preferred answer style in one sentence.",
)
print(first_answer)

I understand and will provide concise explanations paired with one practical example for your requests.


In [ ]:
second_answer = chat_turn(
    state,
    "Now explain the purpose of a tool registry.",
)
print(second_answer)

print()
print("Provider interaction ID:", state.previous_interaction_id)
print("Locally recorded messages:", len(state.messages))

A tool registry is a centralized system used by AI applications to store, manage, and discover available functions or tools that an agent can execute. 

**Example:** 
An AI assistant uses a tool registry to look up and select a `weather_forecast` function when a user asks about the temperature outside.

Provider interaction ID: v1_ChdpQjIyYXJPLU9mREVnOFVQbE9LQm1RWRIXT0I2MmF0NjNMOVdvZzhVUDA0dmEyUUk
Locally recorded messages: 5


Gemini can use the provider-managed interaction history, while the application reinjects approved structured preferences on each turn. These mechanisms solve different problems: interaction continuity preserves conversational context, and structured memory gives the application explicit control over selected facts.

## 7. Inspect the local conversation

The local history can support debugging, user-visible transcripts, context selection, and evaluation. Sensitive content should be redacted or excluded before logging in production.

In [ ]:
def display_recent_messages(
    state: ConversationState,
    limit: int = 6,
) -> None:
    for message in state.recent_messages(limit):
        print(f"[{message.role.upper()}] {message.content}")


display_recent_messages(state)

[USER] I prefer concise explanations with one practical example.
[USER] Acknowledge my preferred answer style in one sentence.
[ASSISTANT] I understand and will provide concise explanations paired with one practical example for your requests.
[USER] Now explain the purpose of a tool registry.
[ASSISTANT] A tool registry is a centralized system used by AI applications to store, manage, and discover available functions or tools that an agent can execute. 

**Example:** 
An AI assistant uses a tool registry to look up and select a `weather_forecast` function when a user asks about the temperature outside.


## 8. Context grows and must be managed

Sending an ever-growing transcript increases tokens, latency, and cost. Older details may also distract the model from the current goal. A simple first policy keeps a recent window and summarizes or retrieves older information separately.

Character count is not an exact token counter, but a rough estimate is sufficient to visualize growth in this tutorial.

In [ ]:
def approximate_tokens(text: str) -> int:
    return max(1, round(len(text) / 4))


def context_report(state: ConversationState, recent_limit: int = 6) -> dict[str, int]:
    all_text = "\n".join(message.content for message in state.messages)
    recent_text = "\n".join(
        message.content for message in state.recent_messages(recent_limit)
    )

    return {
        "message_count": len(state.messages),
        "approximate_all_tokens": approximate_tokens(all_text),
        "recent_message_count": len(state.recent_messages(recent_limit)),
        "approximate_recent_tokens": approximate_tokens(recent_text),
    }


print(context_report(state))

{'message_count': 5, 'approximate_all_tokens': 141, 'recent_message_count': 5, 'approximate_recent_tokens': 141}


For billing and hard context limits, use the model provider's actual token-usage data or tokenizer rather than this approximation.

## 9. Summarize older messages locally

A summary is a lossy compression of conversation history. It can preserve decisions and preferences while dropping exact wording, but important details may disappear or drift. The implementation below demonstrates the state transition with a deterministic summary; a real system could use a model and then evaluate the summary.

In [ ]:
def summarize_older_messages(
    state: ConversationState,
    keep_recent: int = 4,
) -> None:
    if keep_recent < 1:
        raise ValueError("keep_recent must be at least 1")

    if len(state.messages) <= keep_recent:
        return

    older_messages = state.messages[:-keep_recent]
    recent_messages = state.messages[-keep_recent:]

    summary_lines = [
        f"{message.role}: {message.content[:120]}"
        for message in older_messages
    ]

    state.summary = " | ".join(summary_lines)
    state.messages = recent_messages


demo_state = ConversationState()
for index in range(1, 9):
    role: Role = "user" if index % 2 else "assistant"
    demo_state.add_message(role, f"Demonstration message number {index}")

print("Before:", len(demo_state.messages), "messages")
summarize_older_messages(demo_state, keep_recent=4)
print("After :", len(demo_state.messages), "messages")
print("Summary:", demo_state.summary)

Before: 8 messages
After : 4 messages
Summary: user: Demonstration message number 1 | assistant: Demonstration message number 2 | user: Demonstration message number 3 | assistant: Demonstration message number 4


The summary should not silently replace the authoritative record when exact history matters. Depending on the application, raw messages may need to remain in protected storage for audit while only a bounded context is sent to the model.

## 10. Reset and forgetting behavior

Resetting a provider thread and deleting application memory are separate operations. The following examples make that distinction explicit.

In [ ]:
reset_demo = ConversationState()
reset_demo.add_message("user", "My preferred language is English.")
reset_demo.remember("language", "English")
reset_demo.previous_interaction_id = "example-interaction-id"

reset_demo.reset_thread(keep_preferences=True)
print("After thread reset:", reset_demo)

reset_demo.forget("language")
print("After forgetting preference:", reset_demo.preferences)

After thread reset: ConversationState(conversation_id='b813d64a8d1b411d98f54597aa448065', previous_interaction_id=None, messages=[], preferences={'language': 'English'}, summary=None, metadata={})
After forgetting preference: {}


A production deletion request must also cover any durable database records, caches, backups, analytics copies, and provider-side stored interactions governed by the application's retention policy. Clearing one Python object is only the in-memory portion of that process.

## 11. Exercise: build a bounded context payload

Implement `build_context_payload` so it returns:

- the structured memory context;
- the optional summary;
- no more than the most recent `message_limit` messages; and
- an approximate token count for the assembled payload.

The returned dictionary should be serializable and should not modify the state.

In [ ]:
def build_context_payload(
    state: ConversationState,
    message_limit: int = 6,
) -> dict[str, Any]:
    # Build and return a bounded, serializable context payload.
    raise NotImplementedError

<details>
<summary><strong>Reveal one possible solution</strong></summary>

```python
def build_context_payload(
    state: ConversationState,
    message_limit: int = 6,
) -> dict[str, Any]:
    recent_messages = [
        {
            "role": message.role,
            "content": message.content,
            "created_at": message.created_at,
        }
        for message in state.recent_messages(message_limit)
    ]

    payload = {
        "memory_context": build_memory_context(state),
        "summary": state.summary,
        "recent_messages": recent_messages,
    }

    serialized_text = str(payload)
    payload["approximate_tokens"] = approximate_tokens(serialized_text)
    return payload
```

</details>

## Notebook summary

We separated temporary run state, conversation history, structured memory, and persistent storage. The application now owns an inspectable `ConversationState`, while Gemini's `previous_interaction_id` maintains provider-side conversational continuity. We also introduced recent-message windows, approximate context accounting, summary compression, and explicit reset behavior.

The next notebook will rebuild agent control flow with LangGraph. State will move through named nodes and conditional edges, making branching, checkpointing, and interruption easier to manage than in one growing loop function.

### Reference

- [Gemini Interactions API overview](https://ai.google.dev/gemini-api/docs/interactions-overview)